# 3.6 信息论：一次意外有多少信息

> 3.0 基础课程：先直觉，再符号，再数字代入，再用代码和图形核对。

## Goal

从信息量和熵进入二元/多类交叉熵、CE=H+KL、KL 非对称、Softmax/温度、Normalized Entropy、序列 NLL、perplexity 与 DPO log-ratio。

## Setup 与数据边界

本课不加载用户行为数据。代码中的小数组都是带标签的 **数学教学对象**，只用于验证公式，绝不冒充交互、曝光、标签或行为序列。

In [ ]:
from pathlib import Path
import os, sys
import numpy as np
import matplotlib.pyplot as plt
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARTIFACT_ROOT = Path(os.environ.get("RECSYS_ARTIFACT_ROOT", PROJECT_ROOT)).expanduser().resolve()
sys.path.insert(0, str(PROJECT_ROOT))
TEACHING_OBJECTS_ONLY = True
print({"project_root": str(PROJECT_ROOT), "artifact_root": str(ARTIFACT_ROOT), "kind": "curriculum", "teaching_objects_only": True})

from matplotlib import font_manager

# Matplotlib 默认的 DejaVu Sans 不包含中文字形。优先选择容器中的 Noto CJK
# （镜像已安装 fonts-noto-cjk），其次是宿主机常见中文字体；从字体根源解决，
# 而不是用 warnings.filterwarnings 掩盖 missing glyph 警告。
cjk_candidates = ('Noto Sans CJK SC', 'Noto Sans CJK JP', 'PingFang SC', 'Hiragino Sans GB',
                  'Microsoft YaHei', 'SimHei', 'Arial Unicode MS', 'Songti SC')
installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
cjk_font = next((name for name in cjk_candidates if name in installed_fonts), None)
if cjk_font:
    plt.rcParams['font.sans-serif'] = [cjk_font, 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print('图表字体:', cjk_font or '未找到中文字体（请安装 fonts-noto-cjk）')

## 学习路径

先比较常见与意外消息的信息量；把平均意外程度写成熵；再用交叉熵评估模型概率，最后连接分类、序列生成和偏好优化。

## 符号表

| 符号 | 含义 |
|---|---|
| $p(x)$ | 真实/目标分布 |
| $q(x)$ | 模型给出的分布 |
| $I(x)=-\log_2p(x)$ | 事件的信息量（bit） |
| $H(p)$ | 分布自身的不确定性 |
| $H(p,q)$ | 用模型 q 编码来自 p 的结果的交叉熵 |
| $D_{KL}(p\|q)$ | q 相对 p 多付出的平均编码代价 |

> 本课概率、token 与偏好分数均为数学教学对象，不是假造行为序列。

<a id="information-entropy"></a>
## 1. 信息量与熵

越意外的结果信息量越大：$I(x)=-\log_2p(x)$。概率 1 的必然事件信息量为 0；概率减半，信息量增加 1 bit。熵 $H(p)=-\sum_xp(x)\log_2p(x)$ 是信息量的平均。

### 数字代入 1：公平硬币与偏硬币

公平硬币每面概率 0.5，单次信息量都是 1 bit，熵也是 1 bit。若概率为 $[0.9,0.1]$，熵约 $-(0.9\log_20.9+0.1\log_20.1)=0.469$ bit：结果更可预测。

<a id="cross-entropy-kl"></a>
## 2. 交叉熵、KL 与 `CE = H + KL`

交叉熵 $H(p,q)=-\sum p(x)\log q(x)$ 用模型 q 为真实分布 p 编码。它可拆成 $H(p)+D_{KL}(p\|q)$：真实世界固有的不确定性加上模型不准确的额外代价。KL 非对称，因为“用 q 覆盖 p”的漏项与反方向不同。

二分类标签 $y\in\{0,1\}$ 的交叉熵为 $-[y\log q+(1-y)\log(1-q)]$；one-hot 多分类只保留正确类别的 $-\log q_y$。

### 数字代入 2：自信答错更痛

正确类别是第 1 类。模型 A 给它概率 0.8，损失 $-\ln0.8\approx0.223$；模型 B 只给 0.1，损失 $-\ln0.1\approx2.303$，约大 10 倍。

In [ ]:
# Demo 1：熵、交叉熵和 KL 的数值分解。
def entropy(p):
    p=np.asarray(p,float); return -np.sum(p*np.log2(p))
p=np.array([.7,.2,.1]); q=np.array([.5,.3,.2])
H=entropy(p); CE=-np.sum(p*np.log2(q)); KL=np.sum(p*np.log2(p/q))
print({"H(p)":H,"CE(p,q)":CE,"KL(p||q)":KL,"H+KL":H+KL,
       "KL(q||p)":np.sum(q*np.log2(q/p))})
probs=np.linspace(.01,.99,200)
plt.plot(probs,-np.log(probs)); plt.xlabel("正确类别概率"); plt.ylabel("NLL"); plt.title("答得越自信且越错，惩罚越大"); plt.grid(); plt.show()

<a id="softmax-temperature"></a>
## 3. Softmax、温度与 Normalized Entropy

Softmax 把任意分数变为和为 1 的概率：$q_j=\exp(z_j/\tau)/\sum_k\exp(z_k/\tau)$。小温度放大差异，大温度让分布平坦。实现时先减最大分数，避免指数溢出。

若有 $K$ 类，最大熵为 $\log K$。Normalized Entropy 常写成 $H(q)/\log K$，落在 0 到 1：接近 0 很集中，接近 1 很均匀。具体论文可能用另一归一化口径，比较前必须读定义。

### 数字代入 3：温度

分数 $[2,1]$：$\tau=1$ 时概率约 $[0.731,0.269]$；$\tau=0.5$ 等价于分数差翻倍，约 $[0.881,0.119]$，分布更尖。

<a id="sequence-nll-dpo"></a>
## 4. 序列 NLL、perplexity 与 DPO log-ratio

自回归序列概率是条件概率连乘：$P(y_{1:T})=\prod_tP(y_t\mid y_{<t})$。取负对数后变成可相加的序列 NLL：$-\sum_t\log P(y_t\mid y_{<t})$。平均每 token NLL 为 $\bar L$ 时，perplexity $=e^{\bar L}$，可直觉理解为模型平均在多少个等可能选项间犹豫。

DPO 比较偏好答案 $y^+$ 与不偏好答案 $y^-$ 的 log-probability 差，并减去参考模型的同一差值。log-ratio 抵消共同尺度；正值表示当前策略相对参考更偏向 $y^+$。

### 数字代入 4：序列与偏好

三个正确 token 概率为 $[0.5,0.25,0.8]$，序列概率 $0.1$，NLL $=-\ln0.1=2.303$；平均 NLL $0.768$，perplexity $e^{0.768}\approx2.15$。若策略的 $\log P(y^+)-\log P(y^-)=0.9$，参考模型为 0.2，DPO 的相对偏好 margin 为 $0.7$。

In [ ]:
# Demo 2：温度、Normalized Entropy 与序列 NLL。
scores=np.array([2.,1.,0.])
fig,ax=plt.subplots(figsize=(6,3.3))
for tau in [.3,1.,3.]:
    stable=(scores-scores.max())/tau; soft=np.exp(stable); soft/=soft.sum()
    ne=-np.sum(soft*np.log(soft))/np.log(len(soft))
    ax.plot(range(3),soft,"o-",label=f"tau={tau}, NE={ne:.2f}")
ax.set(xticks=range(3),ylabel="probability",title="Softmax 温度教学对象"); ax.legend(); ax.grid(); plt.show()
token_p=np.array([.5,.25,.8]); nll=-np.log(token_p).sum(); ppl=np.exp(nll/len(token_p))
print({"sequence_probability":token_p.prod(),"NLL":nll,"perplexity":ppl,"DPO_margin":.9-.2})

## 常见误区

- 熵越低模型越好：过度自信且错误的分布也可能低熵。
- KL 是普通距离：它不对称，也不满足距离的全部性质。
- perplexity 可跨词表直接比较：token 划分和数据协议不同会改变数值。
- DPO margin 就是线上提升：它只是训练中的相对偏好信号。

## 算法回链

- [word2vec：负采样二元交叉熵](/notebooks/4_6_word2vec)
- [DSSM：sampled softmax](/notebooks/5_2_dssm)
- [SASRec：next-item NLL](/notebooks/5_4_sasrec)
- [OpenOneRec：序列生成与 DPO](/notebooks/8_2_openonerec_practice)
- [HSTU：序列预测与 Normalized Entropy](/notebooks/8_3_dlrm_hstu_practice)

## Checks

1. 为什么概率减半会多 1 bit 信息？
2. `CE=H+KL` 中哪一项能通过改模型降低？
3. 温度变小为什么让 Softmax 更尖？

In [ ]:
assert np.isclose(CE,H+KL)
assert not np.isclose(KL,np.sum(q*np.log2(q/p)))
assert np.isclose(token_p.prod(),.1)
assert np.isclose(nll,-np.log(.1)) and ppl > 1
print("PASS：熵、CE/KL、Softmax、序列 NLL 与 DPO margin 检查通过。")

## Next Steps

下一课进入 3.7 优化，学习怎样用 mini-batch、学习率、正则化和早停稳定地降低这些损失。